# Session 8 — Tool Use and Function Calling

LLMs are excellent at reasoning but cannot act on the world directly. Tool use (also called function calling) gives the model a way to request external actions — fetching live data, running calculations, searching a document store — and incorporate the results into its answer.

This notebook covers:
1. Anatomy of a tool definition (JSON schema)
2. A custom weather tool — the classic first example
3. The agentic loop step by step
4. Built-in tools: `web_search` and `code_interpreter`
5. Parallel tool calls — two tools in one turn
6. RAG as a tool call — the model decides when to retrieve

## Learning Goals

- understand the JSON schema format for tool definitions
- implement the four-step agentic loop (request → detect → execute → feed back)
- use built-in OpenAI tools without writing a Python function
- define multiple tools and handle parallel calls
- refactor the RAG pipeline from notebook 04 so the model controls retrieval

In [ ]:
import json
import os
import requests
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_ORG_ID = os.getenv("OPENAI_ORG_ID")
OPENAI_PROJECT_ID = os.getenv("OPENAI_PROJECT_ID")

print("OpenAI key present:", bool(OPENAI_API_KEY))


def make_openai_client(api_key=None, base_url=None):
    kwargs = {}
    if api_key:
        kwargs["api_key"] = api_key
    if base_url:
        kwargs["base_url"] = base_url
    if api_key == OPENAI_API_KEY and not base_url:
        if OPENAI_ORG_ID:
            kwargs["organization"] = OPENAI_ORG_ID
        if OPENAI_PROJECT_ID:
            kwargs["project"] = OPENAI_PROJECT_ID
    return OpenAI(**kwargs)


def require_env(name: str):
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Missing required environment variable: {name}")
    return value


def session8_dir() -> Path:
    cwd = Path.cwd()
    if cwd.name == "notebooks":
        return cwd.parent
    candidate = cwd / "material" / "Session 8"
    if candidate.exists():
        return candidate
    return cwd

## 1. Anatomy of a Tool Definition

Every tool is a Python dict that follows the OpenAI JSON schema format. The model reads `description` to decide when to call the tool, and `parameters` to know what arguments to pass.

In [ ]:
# Annotated tool schema — nothing executes here, just for reading

weather_tool = {
    "type": "function",           # always "function" for custom tools
    "name": "get_weather",        # must match the Python function name you will call
    "description": (
        "Returns current weather conditions for a city. "
        "Call this whenever the user asks about weather or temperature."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "City name, e.g. Sydney, Paris, New York",
            },
            "units": {
                "type": "string",
                "enum": ["celsius", "fahrenheit"],
                "description": "Temperature unit to return.",
            },
        },
        "required": ["location", "units"],
        "additionalProperties": False,
    },
    "strict": True,   # model must fill all required fields; no extra fields allowed
}

print(json.dumps(weather_tool, indent=2))

## 2. Custom Tool: Weather

`get_weather` calls the free wttr.in API — no API key needed. The function signature must match the `name` and `parameters` in the schema above.

In [ ]:
def get_weather(location: str, units: str = "celsius") -> dict:
    """Fetch current weather from wttr.in (free, no API key required)."""
    unit_char = "m" if units == "celsius" else "u"
    url = f"https://wttr.in/{requests.utils.quote(location)}?format=j1&{unit_char}"
    try:
        resp = requests.get(url, timeout=5)
        resp.raise_for_status()
        data = resp.json()
        current = data["current_condition"][0]
        temp = current["temp_C"] if units == "celsius" else current["temp_F"]
        desc = current["weatherDesc"][0]["value"]
        return {
            "location": location,
            "temperature": int(temp),
            "units": units,
            "description": desc,
        }
    except Exception as e:
        return {"error": str(e), "location": location}


# Quick smoke test — confirm the function works before wiring it to the model
result = get_weather("Sydney", "celsius")
print(result)

## 3. The Agentic Loop

Tool use works in four steps:

1. **Request** — send the user message plus the tool list to the model
2. **Detect** — check if the model returned a `function_call` in `response.output`
3. **Execute** — call the matching Python function with the model's arguments
4. **Feed back** — append the result as `function_call_output` and call the model again

The model then produces a final answer that incorporates the tool result.

### Step 1 — Send the request with tools

In [ ]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = make_openai_client(api_key=require_env("OPENAI_API_KEY"))

input_messages = [
    {"role": "user", "content": "What is the weather like in Tokyo right now? Use celsius."}
]

response = client.responses.create(
    model="gpt-4o-mini",
    tools=[weather_tool],
    input=input_messages,
    tool_choice="auto",   # model decides whether to call a tool
)

print("Output types:", [item.type for item in response.output])

### Step 2 — Detect the function call

The model signals it wants to call a tool by including a `function_call` item in `response.output`. We inspect the output list and find it.

In [ ]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

tool_call = None
for item in response.output:
    if item.type == "function_call":
        tool_call = item
        break

if tool_call:
    print("Function to call:", tool_call.name)
    print("Arguments (JSON string):", tool_call.arguments)
    print("Call ID:", tool_call.call_id)
else:
    print("Model did not request a tool call.")

### Step 3 — Execute the function locally

We parse the JSON arguments and call the matching Python function.

In [ ]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

if tool_call:
    args = json.loads(tool_call.arguments)
    print("Parsed arguments:", args)

    # Dispatch: call the function whose name matches tool_call.name
    tool_result = get_weather(**args)
    print("Tool result:", tool_result)

### Step 4 — Feed the result back and get the final answer

We append the model's tool call output and a `function_call_output` message, then call the model again.

In [ ]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

if tool_call:
    # Append the model's tool call items to the conversation
    input_messages = input_messages + list(response.output)

    # Append our tool result
    input_messages.append({
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(tool_result),
    })

    # Second model call — now it has the tool result
    final_response = client.responses.create(
        model="gpt-4o-mini",
        tools=[weather_tool],
        input=input_messages,
        tool_choice="auto",
    )

    display(Markdown(final_response.output_text))

### Clean Helper: `run_tool_loop`

The four steps above always follow the same pattern. We can wrap them in a reusable function.

In [ ]:
# Map of tool name -> callable Python function
TOOL_REGISTRY = {
    "get_weather": get_weather,
}


def run_tool_loop(
    user_message: str,
    tools: list,
    registry: dict,
    model: str = "gpt-4o-mini",
    max_rounds: int = 5,
) -> str:
    """
    Run the agentic tool loop until the model stops calling tools or max_rounds is reached.
    Returns the final text response.
    """
    client = make_openai_client(api_key=require_env("OPENAI_API_KEY"))
    messages = [{"role": "user", "content": user_message}]

    for _ in range(max_rounds):
        response = client.responses.create(
            model=model,
            tools=tools,
            input=messages,
            tool_choice="auto",
        )

        calls = [item for item in response.output if item.type == "function_call"]

        if not calls:
            return response.output_text

        messages = messages + list(response.output)

        for call in calls:
            fn = registry.get(call.name)
            if fn is None:
                result = {"error": f"Unknown tool: {call.name}"}
            else:
                result = fn(**json.loads(call.arguments))

            messages.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(result),
            })

    return "Max tool rounds reached without a final answer." 

In [ ]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

answer = run_tool_loop(
    user_message="What is the weather in London and Paris? Give me celsius for both.",
    tools=[weather_tool],
    registry=TOOL_REGISTRY,
)
display(Markdown(answer))

## 4. Built-in Tools

OpenAI provides built-in tools you can enable with a single dict — no Python function required. The model calls them server-side and returns the result inline.

The two most useful for teaching:
- `web_search` — live web retrieval
- `code_interpreter` — the model writes and executes Python in a sandbox

### `web_search`

Just declare `{"type": "web_search"}` in the tools list. No schema needed, no Python function to write.

In [ ]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = make_openai_client(api_key=require_env("OPENAI_API_KEY"))

web_response = client.responses.create(
    model="gpt-4o-mini",
    tools=[{"type": "web_search"}],
    input="What were the main AI announcements from the last week?",
)

display(Markdown(web_response.output_text))

### `code_interpreter`

The model writes Python, executes it in a sandboxed environment, and returns the result. Useful for maths, data transformation, or anything a small script can solve.

In [ ]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = make_openai_client(api_key=require_env("OPENAI_API_KEY"))

code_response = client.responses.create(
    model="gpt-4o-mini",
    tools=[{"type": "code_interpreter", "container": {"type": "auto"}}],
    input=(
        "Calculate the compound interest on $10,000 invested at 7% annual rate "
        "for 20 years. Show the formula and the result."
    ),
)

display(Markdown(code_response.output_text))

### Key Difference from Custom Tools

| | Custom tool | Built-in tool |
|---|---|---|
| Python function needed? | Yes | No |
| Runs where? | Your machine | OpenAI servers |
| Can access internet? | If your function does | Yes (`web_search`) |
| Can run code? | If your function does | Yes (`code_interpreter`) |

## 5. Parallel Tool Calls

When multiple tools are registered, the model may call several in a single turn. This is called parallel tool calling and it saves a round-trip for questions that need two independent lookups.

In [ ]:
def get_current_date() -> dict:
    """Return today's date — no external API needed."""
    from datetime import date
    today = date.today()
    return {
        "date": today.isoformat(),
        "day_of_week": today.strftime("%A"),
    }


date_tool = {
    "type": "function",
    "name": "get_current_date",
    "description": "Returns today's date and day of the week. Call this when the user asks what day or date it is.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False,
    },
    "strict": True,
}

# Smoke test
print(get_current_date())

In [ ]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

TOOL_REGISTRY_V2 = {
    "get_weather": get_weather,
    "get_current_date": get_current_date,
}

answer = run_tool_loop(
    user_message=(
        "What day is it today, and what is the weather in Melbourne right now in celsius?"
    ),
    tools=[weather_tool, date_tool],
    registry=TOOL_REGISTRY_V2,
)

display(Markdown(answer))

The model may issue both `get_weather` and `get_current_date` calls in the same response turn. The `run_tool_loop` helper handles this naturally because it loops over all `function_call` items before making the next model call.